# Exp48 — RAFT-Matched retrieval comparison on the frozen 24-condition benchmark

**Kernel:** `Python 3.12 (Stock Regime)`

This notebook adds the remaining direct retrieval baseline:

> Han et al., **Retrieval Augmented Time Series Forecasting (RAFT)**, ICML 2025.

The official RAFT implementation (`archon159/RAFT`, `layers/Retrieval.py`) uses:

- three temporal granularities \(g\in\{4,2,1\}\) when `n_period=3`;
- non-overlapping average pooling at each granularity, repeated back to the original length;
- endpoint-offset removal within each channel;
- normalized correlation after flattening the full multivariate window;
- `topm=20` candidates per granularity;
- softmax temperature \(\tau=0.1\).

The frozen PRR retrieval benchmark uses \(L=96\), memory stride 8, Top-10, and
AnalogFutureMSE.  Therefore this notebook evaluates **RAFT-Matched** under the exact
same query manifests, temporal-admissibility rule, normalization, memory stride, and
Top-10 metric.

### Important scope

This is a **matched retrieval-rule comparison**, not a reproduction of RAFT's end-to-end
forecasting benchmark.  RAFT's official code produces one retrieval-weighted prediction
per temporal granularity and lets the forecasting architecture consume those retrievals;
it does **not** define one unique Top-10 history list across the three granularities.

We therefore report two transparent single-list projections:

1. **RAFT-MS@10 (primary matched projection)**  
   For each granularity, keep the official Top-20 and temperature-0.1 softmax weights.
   Average candidate probability mass across the three granularities (zero for a candidate
   absent from a scale's Top-20), then take the final Top-10 histories.

2. **RAFT-ValScale@10 (strong validation-selected control)**  
   On validation only, choose the best single official granularity from \(g\in\{1,2,4\}\)
   by AnalogFutureMSE; freeze that choice and evaluate its Top-10 on test.

The second control prevents the multiscale projection choice from artificially weakening RAFT.

All six datasets in the present paper (ETTh1, Weather, Electricity, Traffic, Exchange,
Solar) are among the ten datasets evaluated by RAFT, so the full 24-condition comparison
is within RAFT's benchmark dataset family.


In [1]:
# Cell 1 — Stock Regime kernel guard and environment
import sys, os, json, time, gc, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
print("PyTorch          :", torch.__version__)
print("NumPy            :", np.__version__)
print("CUDA available   :", torch.cuda.is_available())

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Wrong Jupyter kernel. Select 'Python 3.12 (Stock Regime)' / "
        "'Stock Regime' before running this notebook."
    )

if not torch.cuda.is_available():
    raise RuntimeError("This matched RAFT experiment is intended for the GPU Stock Regime kernel.")

DEVICE = torch.device("cuda")
print("GPU              :", torch.cuda.get_device_name(0))
print("[PASS] Stock Regime kernel detected")


Python executable: /data/envs/stock_regime/bin/python
Python version   : 3.12.13
PyTorch          : 2.12.1+cu126
NumPy            : 2.4.6
CUDA available   : True
GPU              : NVIDIA A100-SXM4-80GB
[PASS] Stock Regime kernel detected


In [2]:
# Cell 2 — configuration
DATASETS = ("ETTh1", "Weather", "Electricity", "Traffic", "Exchange", "Solar")
HORIZONS = (96, 192, 336, 720)

L = 96
MEMORY_STRIDE = 8
TOP_K = 10

# Official RAFT defaults/settings from the public implementation.
RAFT_PERIODS = (4, 2, 1)   # n_period=3 -> [4,2,1]
RAFT_TOPM = 20
RAFT_TEMPERATURE = 0.1

QUERY_CHUNK = 64
FEATURE_CHUNK = 64
METRIC_CHUNK = 128

RUN_SMOKE_TEST = True
RECOMPUTE = False

print("L / stride / TopK         :", L, MEMORY_STRIDE, TOP_K)
print("RAFT periods              :", RAFT_PERIODS)
print("RAFT official topm / temp :", RAFT_TOPM, RAFT_TEMPERATURE)


L / stride / TopK         : 96 8 10
RAFT periods              : (4, 2, 1)
RAFT official topm / temp : 20 0.1


In [3]:
# Cell 3 — locate frozen manifests and project outputs
manifest_candidates = [
    Path("/data/code/which_histories_matter_ext/results/exp33/final_eval_manifest"),
    Path("/data/code/stock_regime_retrieval/strong_forecaster/results/exp33/final_eval_manifest"),
]

for base in [Path("/data/code"), Path("/data")]:
    if base.exists():
        manifest_candidates += list(base.glob("*/results/exp33/final_eval_manifest"))
        manifest_candidates += list(base.glob("*/*/results/exp33/final_eval_manifest"))

MANIFEST_ROOT = next((p for p in manifest_candidates if p.exists()), None)
assert MANIFEST_ROOT is not None, "Exp33 frozen manifests not found."

PROJECT_ROOT = MANIFEST_ROOT.parents[2]
OUT_ROOT = PROJECT_ROOT / "results" / "exp48_raft_matched_retrieval"
CACHE_ROOT = OUT_ROOT / "cache"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "datasets": list(DATASETS),
    "horizons": list(HORIZONS),
    "lookback": L,
    "memory_stride": MEMORY_STRIDE,
    "top_k": TOP_K,
    "raft_periods": list(RAFT_PERIODS),
    "raft_topm": RAFT_TOPM,
    "raft_temperature": RAFT_TEMPERATURE,
    "primary_projection": "mean official top20 softmax probability mass across periods -> top10",
    "strong_control": "validation-selected single official period -> test top10",
    "note": "Matched retrieval-rule comparison; not end-to-end RAFT forecasting reproduction.",
}
(OUT_ROOT / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

print("MANIFEST_ROOT:", MANIFEST_ROOT)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("OUT_ROOT     :", OUT_ROOT)


MANIFEST_ROOT: /data/code/which_histories_matter_ext/results/exp33/final_eval_manifest
PROJECT_ROOT : /data/code/which_histories_matter_ext
OUT_ROOT     : /data/code/which_histories_matter_ext/results/exp48_raft_matched_retrieval


In [4]:
# Cell 4 — dataset loader aligned to the frozen PRR benchmark
EXPECTED = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

CANDIDATES = {
    "ETTh1": [
        "/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv",
        "/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETTh1.csv",
    ],
    "Weather": [
        "/data/Time-Series-Library/dataset/weather/weather.csv",
        "/data/Time-Series-Library_v2/dataset/weather/weather.csv",
        "/data/dataset/weather/weather.csv",
        "/data/dataset/weather.csv",
    ],
    "Electricity": [
        "/data/Time-Series-Library/dataset/electricity/electricity.csv",
        "/data/Time-Series-Library_v2/dataset/electricity/electricity.csv",
        "/data/dataset/electricity/electricity.csv",
        "/data/dataset/electricity.csv",
    ],
    "Traffic": [
        "/data/Time-Series-Library/dataset/traffic/traffic.csv",
        "/data/Time-Series-Library_v2/dataset/traffic/traffic.csv",
        "/data/dataset/traffic/traffic.csv",
        "/data/dataset/traffic.csv",
    ],
    "Exchange": [
        "/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv",
        "/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv",
        "/data/dataset/exchange_rate/exchange_rate.csv",
        "/data/dataset/exchange_rate.csv",
    ],
    "Solar": [
        "/data/Time-Series-Library/dataset/Solar/solar_AL.txt",
        "/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt",
        "/data/dataset/Solar/solar_AL.txt",
        "/data/dataset/solar_AL.txt",
    ],
}

def resolve_dataset(name):
    for s in CANDIDATES[name]:
        p = Path(s)
        if p.is_file():
            return p
    raise FileNotFoundError(f"Dataset not found: {name}")

def clean_numeric(df):
    return (
        df.apply(pd.to_numeric, errors="coerce")
          .dropna(axis=1, how="all")
          .replace([np.inf, -np.inf], np.nan)
          .interpolate(axis=0, limit_direction="both")
          .ffill()
          .bfill()
    )

def load_data(name):
    p = resolve_dataset(name)

    if name == "Solar":
        df = pd.read_csv(p, header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(p, header=None, sep=r"\s+")
        num = clean_numeric(df)
    else:
        df = pd.read_csv(p)
        date_col = next(
            (c for c in df.columns if str(c).lower() in {"date", "datetime", "timestamp", "time"}),
            None,
        )
        val = df.drop(columns=[date_col]) if date_col is not None else df.copy()

        if name in {"Weather", "Electricity"} and "OT" in val.columns:
            val = val[[c for c in val.columns if c != "OT"] + ["OT"]]

        num = clean_numeric(val)

    raw = num.to_numpy(np.float32)
    assert raw.shape[1] == EXPECTED[name], (name, raw.shape)

    n = len(raw)
    if name == "ETTh1":
        assert raw.shape == (17420, 7)
        train_end, val_end = 8640, 11520
    else:
        train_end = int(0.70 * n)
        val_end = n - int(0.20 * n)

    mu = raw[:train_end].mean(0).astype(np.float32)
    sd = raw[:train_end].std(0, ddof=0).astype(np.float32)
    assert np.all(sd > 1e-6), (name, float(sd.min()))

    z = ((raw - mu) / sd).astype(np.float32)

    return {
        "path": p,
        "raw": raw,
        "z": z,
        "n": n,
        "channels": raw.shape[1],
        "train_end": train_end,
        "val_end": val_end,
    }

DATA = {d: load_data(d) for d in DATASETS}

display(pd.DataFrame([
    {
        "dataset": d,
        "rows": DATA[d]["n"],
        "channels": DATA[d]["channels"],
        "train_end": DATA[d]["train_end"],
        "val_end": DATA[d]["val_end"],
        "path": str(DATA[d]["path"]),
    }
    for d in DATASETS
]))


,dataset,rows,channels,train_end,val_end,path
0,ETTh1,17420,7,8640,11520,/data/Time-Series-Library/dataset/ETT-small/ET...
1,Weather,52696,21,36887,42157,/data/Time-Series-Library/dataset/weather/weat...
2,Electricity,26304,321,18412,21044,/data/Time-Series-Library/dataset/electricity/...
3,Traffic,17544,862,12280,14036,/data/Time-Series-Library/dataset/traffic/traf...
4,Exchange,7588,8,5311,6071,/data/Time-Series-Library/dataset/exchange_rat...
5,Solar,52560,137,36792,42048,/data/Time-Series-Library/dataset/Solar/solar_...


In [5]:
# Cell 5 — frozen manifest audit
def load_manifest(dataset, horizon, split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p
    with np.load(p, allow_pickle=False) as zz:
        anchor = np.asarray(zz["anchor"], np.int64)
        channel = np.asarray(zz["channel"], np.int64)

    assert anchor.shape == channel.shape
    assert len(anchor) > 0
    assert channel.min() >= 0 and channel.max() < DATA[dataset]["channels"]
    return anchor, channel

audit = []
for d in DATASETS:
    for h in HORIZONS:
        for split in ("val", "test"):
            a, c = load_manifest(d, h, split)
            audit.append({
                "dataset": d,
                "horizon": h,
                "split": split,
                "pairs": len(a),
                "unique_query_anchors": len(np.unique(a)),
                "unique_channels": len(np.unique(c)),
            })

audit_df = pd.DataFrame(audit)
display(audit_df)
audit_df.to_csv(OUT_ROOT / "manifest_audit.csv", index=False)


,dataset,horizon,split,pairs,unique_query_anchors,unique_channels
0,ETTh1,96,val,4096,2269,7
1,ETTh1,96,test,8192,4593,7
2,ETTh1,192,val,4096,2198,7
3,ETTh1,192,test,8192,4566,7
4,ETTh1,336,val,4096,2145,7
5,ETTh1,336,test,8192,4520,7
6,ETTh1,720,val,4096,1924,7
7,ETTh1,720,test,8192,4320,7
8,Weather,96,val,4096,2889,21
9,Weather,96,test,8192,5773,21


## Exact official RAFT similarity, implemented efficiently

The public RAFT code averages every non-overlapping block of size \(g\), repeats that average
\(g\) times, subtracts the last pooled value channel-wise, flattens the full multivariate window,
globally centers the flattened vector, and applies cosine normalization.

Because every pooled element is repeated **the same number \(g\) of times**, those repeats multiply
all dot products and squared norms by the same factor.  Therefore the normalized correlation is
exactly unchanged if we omit the physical repeat and use the compressed pooled vector directly.

The next cell verifies this equivalence numerically before the large experiment.


In [6]:
# Cell 6 — exact compressed RAFT feature + parity check against literal official decomposition
def raft_feature_torch(z_t, anchors, g):
    """
    Equivalent to official:
      unfold(size=g, step=g).mean -> repeat_interleave(g)
      -> subtract last pooled value per channel
      -> flatten -> global center -> L2 normalize
    but omits the redundant repeat_interleave.
    """
    anchors = torch.as_tensor(anchors, dtype=torch.long, device=z_t.device)
    offs = torch.arange(L, device=z_t.device, dtype=torch.long)
    idx = anchors[:, None] - L + offs[None, :]
    x = z_t[idx]                               # B,L,C
    B, _, C = x.shape
    x = x.reshape(B, L // g, g, C).mean(dim=2)
    x = x - x[:, -1:, :]
    x = x.flatten(start_dim=1)
    x = x - x.mean(dim=1, keepdim=True)
    x = F.normalize(x, dim=1)
    return x

def raft_feature_literal(x, g):
    """
    Small-tensor literal reproduction of the public RAFT decompose_mg +
    periodic_batch_corr preprocessing, used only for the parity test.
    x: B,L,C
    """
    B, _, C = x.shape
    y = x.reshape(B, L // g, g, C).mean(dim=2)
    y = y.repeat_interleave(g, dim=1)
    y = y - y[:, -1:, :]
    y = y.flatten(start_dim=1)
    y = y - y.mean(dim=1, keepdim=True)
    y = F.normalize(y, dim=1)
    return y

torch.manual_seed(0)
toy = torch.randn(7, L, 5, device=DEVICE)

for g in RAFT_PERIODS:
    lit = raft_feature_literal(toy, g)

    # compressed version from equivalent pooled representation
    B, _, C = toy.shape
    cmp = toy.reshape(B, L // g, g, C).mean(dim=2)
    cmp = cmp - cmp[:, -1:, :]
    cmp = cmp.flatten(start_dim=1)
    cmp = cmp - cmp.mean(dim=1, keepdim=True)
    cmp = F.normalize(cmp, dim=1)

    s_lit = lit @ lit.T
    s_cmp = cmp @ cmp.T
    err = float((s_lit - s_cmp).abs().max().item())
    print(f"g={g}: max similarity error = {err:.3e}")
    assert err < 2e-6

print("[PASS] compressed RAFT similarity is numerically equivalent to literal repeat-interleave form")
del toy, lit, cmp
torch.cuda.empty_cache()


g=4: max similarity error = 1.907e-06
g=2: max similarity error = 6.557e-07
g=1: max similarity error = 0.000e+00
[PASS] compressed RAFT similarity is numerically equivalent to literal repeat-interleave form


In [7]:
# Cell 7 — candidate memory and future utilities
def valid_memory_anchors(boundary, horizon):
    # Candidate future must be fully observed before the memory boundary.
    a = np.arange(L, boundary - horizon + 1, MEMORY_STRIDE, dtype=np.int64)
    assert len(a) >= RAFT_TOPM
    assert int(a.max()) + horizon <= boundary
    return a

def future_residuals_variable(z, channels, anchors, horizon):
    channels = np.asarray(channels, np.int64)
    anchors = np.asarray(anchors, np.int64)
    offs = np.arange(horizon, dtype=np.int64)
    fut = z[anchors[:, None] + offs[None, :], channels[:, None]]
    cur = z[anchors - 1, channels]
    return (fut - cur[:, None]).astype(np.float32)

def cache_path(dataset, horizon, split):
    return CACHE_ROOT / f"{dataset}_H{horizon}_{split}_raftmatched.npz"

def softmax_np(x, temp):
    x = np.asarray(x, np.float64) / float(temp)
    x -= x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return (e / e.sum(axis=-1, keepdims=True)).astype(np.float32)


In [8]:
# Cell 8 — obtain official Top-20 per scale for each unique query anchor
def get_scale_topm(dataset, horizon, split, g, unique_qanchors):
    z = DATA[dataset]["z"]
    boundary = DATA[dataset]["train_end"] if split == "val" else DATA[dataset]["val_end"]
    mem_anchors = valid_memory_anchors(boundary, horizon)

    z_t = torch.from_numpy(z).to(DEVICE, non_blocking=True)

    # Candidate features.  At Traffic/g=1 this is the largest tensor but is still
    # comfortably below A100 80GB memory.
    cand_parts = []
    with torch.no_grad():
        for left in range(0, len(mem_anchors), FEATURE_CHUNK):
            aa = mem_anchors[left:left + FEATURE_CHUNK]
            cand_parts.append(raft_feature_torch(z_t, aa, g))
        cand = torch.cat(cand_parts, dim=0)
        del cand_parts

    top_idx_all = np.empty((len(unique_qanchors), RAFT_TOPM), np.int64)
    top_sim_all = np.empty((len(unique_qanchors), RAFT_TOPM), np.float32)

    with torch.no_grad():
        for left in range(0, len(unique_qanchors), QUERY_CHUNK):
            qa = unique_qanchors[left:left + QUERY_CHUNK]
            qf = raft_feature_torch(z_t, qa, g)
            sim = qf @ cand.T

            vals, idx = torch.topk(sim, k=RAFT_TOPM, dim=1, largest=True, sorted=True)
            top_idx_all[left:left + len(qa)] = idx.cpu().numpy()
            top_sim_all[left:left + len(qa)] = vals.float().cpu().numpy()

            del qf, sim, vals, idx

    del cand, z_t
    torch.cuda.empty_cache()

    return mem_anchors, top_idx_all, top_sim_all

def aggregate_multiscale_top10(top_idx_by_g, top_sim_by_g):
    """
    Single-list matched projection:
      official Top-20 per scale -> official temp=.1 softmax within each scale
      -> average probability mass across scales -> final Top-10.
    """
    U = next(iter(top_idx_by_g.values())).shape[0]
    out = np.empty((U, TOP_K), np.int64)
    union_size = np.empty(U, np.int16)

    for u in range(U):
        mass = {}
        for g in RAFT_PERIODS:
            idx = top_idx_by_g[g][u]
            prob = softmax_np(top_sim_by_g[g][u][None, :], RAFT_TEMPERATURE)[0]
            for j, p in zip(idx.tolist(), prob.tolist()):
                mass[j] = mass.get(j, 0.0) + p / len(RAFT_PERIODS)

        items = sorted(mass.items(), key=lambda kv: (-kv[1], kv[0]))
        union_size[u] = len(items)
        assert len(items) >= TOP_K
        out[u] = np.asarray([j for j, _ in items[:TOP_K]], np.int64)

    return out, union_size


In [9]:
# Cell 9 — vectorized AnalogFutureMSE for channel-specific frozen query pairs
def evaluate_rankings(dataset, horizon, qa, qc, unique_qanchors, mem_anchors, rank_idx):
    """
    rank_idx: [num_unique_query_anchors, TOP_K] indices into mem_anchors.
    Query ranking is multivariate and therefore shared by all target channels with
    the same query anchor; AnalogFutureMSE is evaluated on the frozen target channel.
    """
    z = DATA[dataset]["z"]
    qmap = {int(a): i for i, a in enumerate(unique_qanchors)}
    urow = np.asarray([qmap[int(a)] for a in qa], np.int64)

    per_pair = np.empty(len(qa), np.float32)
    rf_pair = np.empty(len(qa), np.float32)
    offs = np.arange(horizon, dtype=np.int64)

    for left in range(0, len(qa), METRIC_CHUNK):
        sl = slice(left, min(left + METRIC_CHUNK, len(qa)))
        a = qa[sl]
        c = qc[sl]
        r = urow[sl]

        cand_a = mem_anchors[rank_idx[r]]  # B,K

        # Candidate future, endpoint-relative, on the same target channel as the query.
        t_idx = cand_a[:, :, None] + offs[None, None, :]
        ch_idx = c[:, None, None]
        cand_fut = z[t_idx, ch_idx]
        cand_cur = z[cand_a - 1, c[:, None]]
        cand_rel = cand_fut - cand_cur[:, :, None]

        # Query target future.
        q_fut = z[a[:, None] + offs[None, :], c[:, None]]
        q_cur = z[a - 1, c]
        q_rel = q_fut - q_cur[:, None]

        dist = ((cand_rel - q_rel[:, None, :]) ** 2).mean(axis=2)
        per_pair[sl] = dist.mean(axis=1)

        mean_rel = cand_rel.mean(axis=1)
        rf_pair[sl] = ((mean_rel - q_rel) ** 2).mean(axis=1)

    return per_pair, rf_pair


In [10]:
# Cell 10 — one condition/split
def evaluate_condition_split(dataset, horizon, split):
    p = cache_path(dataset, horizon, split)

    qa, qc = load_manifest(dataset, horizon, split)

    if p.exists() and not RECOMPUTE:
        with np.load(p, allow_pickle=False) as zz:
            ca = np.asarray(zz["anchor"], np.int64)
            cc = np.asarray(zz["channel"], np.int64)
            if np.array_equal(ca, qa) and np.array_equal(cc, qc):
                return {
                    "dataset": dataset,
                    "horizon": horizon,
                    "split": split,
                    "pairs": len(qa),
                    "unique_query_anchors": int(np.asarray(zz["unique_query_anchors"]).item()),
                    "raft_ms_afmse": float(np.asarray(zz["raft_ms_afmse"]).mean()),
                    "raft_ms_rfmse": float(np.asarray(zz["raft_ms_rfmse"]).mean()),
                    "g1_afmse": float(np.asarray(zz["g1_afmse"]).mean()),
                    "g2_afmse": float(np.asarray(zz["g2_afmse"]).mean()),
                    "g4_afmse": float(np.asarray(zz["g4_afmse"]).mean()),
                    "mean_multiscale_union": float(np.asarray(zz["mean_multiscale_union"]).item()),
                    "seconds": 0.0,
                    "cached": True,
                }

    t0 = time.perf_counter()
    unique_qanchors = np.unique(qa)

    top_idx = {}
    top_sim = {}
    mem_ref = None

    for g in RAFT_PERIODS:
        print(
            f"RAFT | {dataset:11s} H={horizon:3d} {split:4s} | "
            f"g={g} | unique queries={len(unique_qanchors)}"
        )
        mem, ii, ss = get_scale_topm(dataset, horizon, split, g, unique_qanchors)
        if mem_ref is None:
            mem_ref = mem
        else:
            assert np.array_equal(mem_ref, mem)
        top_idx[g] = ii
        top_sim[g] = ss

    ms_rank, union_size = aggregate_multiscale_top10(top_idx, top_sim)

    # Per-scale Top-10s use the first 10 of the official sorted Top-20.
    rank_g = {g: top_idx[g][:, :TOP_K].copy() for g in RAFT_PERIODS}

    ms_af, ms_rf = evaluate_rankings(
        dataset, horizon, qa, qc, unique_qanchors, mem_ref, ms_rank
    )

    scale_af = {}
    scale_rf = {}
    for g in RAFT_PERIODS:
        af, rf = evaluate_rankings(
            dataset, horizon, qa, qc, unique_qanchors, mem_ref, rank_g[g]
        )
        scale_af[g] = af
        scale_rf[g] = rf

    elapsed = time.perf_counter() - t0

    np.savez_compressed(
        p,
        anchor=qa.astype(np.int64),
        channel=qc.astype(np.int16),
        unique_query_anchors=np.asarray(len(unique_qanchors), np.int64),
        memory_anchors=mem_ref.astype(np.int64),
        raft_ms_afmse=ms_af.astype(np.float32),
        raft_ms_rfmse=ms_rf.astype(np.float32),
        g1_afmse=scale_af[1].astype(np.float32),
        g2_afmse=scale_af[2].astype(np.float32),
        g4_afmse=scale_af[4].astype(np.float32),
        g1_rfmse=scale_rf[1].astype(np.float32),
        g2_rfmse=scale_rf[2].astype(np.float32),
        g4_rfmse=scale_rf[4].astype(np.float32),
        multiscale_union_size=union_size.astype(np.int16),
        mean_multiscale_union=np.asarray(float(union_size.mean()), np.float32),
        memory_stride=np.asarray(MEMORY_STRIDE, np.int64),
        topm=np.asarray(RAFT_TOPM, np.int64),
        temperature=np.asarray(RAFT_TEMPERATURE, np.float32),
    )

    return {
        "dataset": dataset,
        "horizon": horizon,
        "split": split,
        "pairs": len(qa),
        "unique_query_anchors": len(unique_qanchors),
        "raft_ms_afmse": float(ms_af.mean()),
        "raft_ms_rfmse": float(ms_rf.mean()),
        "g1_afmse": float(scale_af[1].mean()),
        "g2_afmse": float(scale_af[2].mean()),
        "g4_afmse": float(scale_af[4].mean()),
        "mean_multiscale_union": float(union_size.mean()),
        "seconds": float(elapsed),
        "cached": False,
    }


In [11]:
# Cell 11 — smoke test
if RUN_SMOKE_TEST:
    print("\n" + "=" * 100)
    print("SMOKE TEST: ETTh1 H=96 validation")
    print("=" * 100)
    smoke = evaluate_condition_split("ETTh1", 96, "val")
    print(json.dumps(smoke, indent=2))
    assert np.isfinite(smoke["raft_ms_afmse"])
    print("[PASS] RAFT matched smoke test completed")



SMOKE TEST: ETTh1 H=96 validation
RAFT | ETTh1       H= 96 val  | g=4 | unique queries=2269
RAFT | ETTh1       H= 96 val  | g=2 | unique queries=2269
RAFT | ETTh1       H= 96 val  | g=1 | unique queries=2269
{
  "dataset": "ETTh1",
  "horizon": 96,
  "split": "val",
  "pairs": 4096,
  "unique_query_anchors": 2269,
  "raft_ms_afmse": 1.457352638244629,
  "raft_ms_rfmse": 1.059533953666687,
  "g1_afmse": 1.438315987586975,
  "g2_afmse": 1.4617352485656738,
  "g4_afmse": 1.519669532775879,
  "mean_multiscale_union": 30.914940502423974,
  "seconds": 0.40575394686311483,
  "cached": false
}
[PASS] RAFT matched smoke test completed


In [12]:
# Cell 12 — full 24 conditions, validation + test
rows = []

for d in DATASETS:
    for h in HORIZONS:
        print("\n" + "=" * 108)
        print("RAFT MATCHED:", d, "H=", h)
        print("=" * 108)

        rv = evaluate_condition_split(d, h, "val")
        rt = evaluate_condition_split(d, h, "test")
        rows.extend([rv, rt])

        print(
            f"VAL  MS={rv['raft_ms_afmse']:.6f} "
            f"g1={rv['g1_afmse']:.6f} g2={rv['g2_afmse']:.6f} g4={rv['g4_afmse']:.6f}"
        )
        print(
            f"TEST MS={rt['raft_ms_afmse']:.6f} "
            f"g1={rt['g1_afmse']:.6f} g2={rt['g2_afmse']:.6f} g4={rt['g4_afmse']:.6f}"
        )

        gc.collect()
        torch.cuda.empty_cache()

raw = pd.DataFrame(rows).sort_values(["dataset", "horizon", "split"]).reset_index(drop=True)
assert len(raw) == 48
display(raw)
raw.to_csv(OUT_ROOT / "raft_matched_val_test_raw.csv", index=False)



RAFT MATCHED: ETTh1 H= 96
RAFT | ETTh1       H= 96 test | g=4 | unique queries=4593
RAFT | ETTh1       H= 96 test | g=2 | unique queries=4593
RAFT | ETTh1       H= 96 test | g=1 | unique queries=4593
VAL  MS=1.457353 g1=1.438316 g2=1.461735 g4=1.519670
TEST MS=1.439627 g1=1.417994 g2=1.439682 g4=1.540221

RAFT MATCHED: ETTh1 H= 192
RAFT | ETTh1       H=192 val  | g=4 | unique queries=2198
RAFT | ETTh1       H=192 val  | g=2 | unique queries=2198
RAFT | ETTh1       H=192 val  | g=1 | unique queries=2198
RAFT | ETTh1       H=192 test | g=4 | unique queries=4566
RAFT | ETTh1       H=192 test | g=2 | unique queries=4566
RAFT | ETTh1       H=192 test | g=1 | unique queries=4566
VAL  MS=1.785716 g1=1.764996 g2=1.794087 g4=1.848967
TEST MS=1.613701 g1=1.602331 g2=1.632369 g4=1.698162

RAFT MATCHED: ETTh1 H= 336
RAFT | ETTh1       H=336 val  | g=4 | unique queries=2145
RAFT | ETTh1       H=336 val  | g=2 | unique queries=2145
RAFT | ETTh1       H=336 val  | g=1 | unique queries=2145
RAFT | ET

,dataset,horizon,split,pairs,unique_query_anchors,raft_ms_afmse,raft_ms_rfmse,g1_afmse,g2_afmse,g4_afmse,mean_multiscale_union,seconds,cached
0,ETTh1,96,test,8192,4593,1.439627,0.925646,1.417994,1.439682,1.540221,32.119530,0.672332,False
1,ETTh1,96,val,4096,2269,1.457353,1.059534,1.438316,1.461735,1.519670,30.914940,0.000000,True
2,ETTh1,192,test,8192,4566,1.613701,0.987824,1.602331,1.632369,1.698162,32.081253,0.905931,False
3,ETTh1,192,val,4096,2198,1.785716,1.350553,1.764996,1.794087,1.848967,30.625569,0.453273,False
4,ETTh1,336,test,8192,4520,1.733000,1.031337,1.741645,1.751988,1.825369,32.012832,1.297466,False
5,ETTh1,336,val,4096,2145,2.173145,1.685995,2.150057,2.172072,2.247648,30.592075,0.636326,False
6,ETTh1,720,test,8192,4320,1.978460,1.156218,1.980019,1.985554,2.073860,32.006944,2.327100,False
7,ETTh1,720,val,4096,1924,2.488045,1.990933,2.478546,2.495787,2.581240,30.353430,1.159798,False
8,Electricity,96,test,8192,4140,1.195281,0.979646,1.208503,1.198295,1.198448,29.244686,0.786347,False
9,Electricity,96,val,4096,2019,1.146747,0.936501,1.151793,1.139951,1.161979,29.153541,0.496844,False


In [13]:
# Cell 13 — validation-selected strongest single RAFT granularity
val = raw[raw["split"] == "val"].copy()
test = raw[raw["split"] == "test"].copy()

period_cols = {1: "g1_afmse", 2: "g2_afmse", 4: "g4_afmse"}

selected = []
for r in val.itertuples(index=False):
    values = {g: float(getattr(r, col)) for g, col in period_cols.items()}
    best_g = min(values, key=values.get)
    selected.append({
        "dataset": r.dataset,
        "horizon": int(r.horizon),
        "selected_g": int(best_g),
        "val_afmse": float(values[best_g]),
    })

sel = pd.DataFrame(selected)

test_sel = test.merge(sel, on=["dataset", "horizon"], how="inner")
assert len(test_sel) == 24

def picked_test_value(row):
    return float(row[period_cols[int(row["selected_g"])]])

test_sel["raft_valscale_afmse"] = test_sel.apply(picked_test_value, axis=1)

display(test_sel[[
    "dataset","horizon","selected_g","val_afmse",
    "raft_ms_afmse","raft_valscale_afmse",
]])

test_sel.to_csv(OUT_ROOT / "raft_matched_test_with_valscale.csv", index=False)


,dataset,horizon,selected_g,val_afmse,raft_ms_afmse,raft_valscale_afmse
0,ETTh1,96,1,1.438316,1.439627,1.417994
1,ETTh1,192,1,1.764996,1.613701,1.602331
2,ETTh1,336,1,2.150057,1.733000,1.741645
3,ETTh1,720,1,2.478546,1.978460,1.980019
4,Electricity,96,2,1.139951,1.195281,1.198295
5,Electricity,192,2,1.137972,1.232099,1.231629
6,Electricity,336,2,1.224727,1.275571,1.277479
7,Electricity,720,2,1.319731,1.326263,1.327164
8,Exchange,96,1,0.247392,0.206670,0.206644
9,Exchange,192,1,0.504082,0.384629,0.384415


In [14]:
# Cell 14 — exact frozen Pearson / PRR / Anchored-L2 references
l2_csv = PROJECT_ROOT / "results" / "anchored_l2_long_horizon_frozen24" / "anchored_l2_test_24conditions.csv"
precise_csv = PROJECT_ROOT / "results" / "stride8_revalidation" / "table1_stride8_revalidated.csv"

if l2_csv.exists():
    l2 = (
        pd.read_csv(l2_csv)[["dataset", "horizon", "anchored_l2_analog_future_mse"]]
          .rename(columns={"anchored_l2_analog_future_mse": "anchored_l2"})
    )
else:
    l2 = pd.DataFrame([
        ("ETTh1",96,1.100985),("ETTh1",192,1.277228),("ETTh1",336,1.446432),("ETTh1",720,1.714012),
        ("Electricity",96,.703011),("Electricity",192,.728439),("Electricity",336,.768702),("Electricity",720,.815970),
        ("Exchange",96,.186254),("Exchange",192,.379211),("Exchange",336,.684537),("Exchange",720,1.865029),
        ("Solar",96,.488054),("Solar",192,.518212),("Solar",336,.533977),("Solar",720,.527471),
        ("Traffic",96,1.420632),("Traffic",192,1.355879),("Traffic",336,1.424405),("Traffic",720,1.595257),
        ("Weather",96,.839593),("Weather",192,.785080),("Weather",336,.594079),("Weather",720,.769931),
    ], columns=["dataset", "horizon", "anchored_l2"])

if precise_csv.exists():
    p = pd.read_csv(precise_csv)
    need = {"dataset", "horizon", "pearson_analog_future_mse", "prr_analog_future_mse"}
    assert need.issubset(p.columns), p.columns
    ref = (
        p[["dataset", "horizon", "pearson_analog_future_mse", "prr_analog_future_mse"]]
        .rename(columns={
            "pearson_analog_future_mse": "pearson",
            "prr_analog_future_mse": "prr",
        })
    )
else:
    ref = pd.DataFrame([
        ("ETTh1",96,1.612267,1.1643),("ETTh1",192,1.817531,1.2229),("ETTh1",336,1.992872,1.4816),("ETTh1",720,2.254092,1.4618),
        ("Weather",96,1.161329,.4316),("Weather",192,1.062809,.5147),("Weather",336,1.290959,.6748),("Weather",720,1.295817,.8130),
        ("Electricity",96,1.179589,.7191),("Electricity",192,1.195698,.7345),("Electricity",336,1.241844,.7648),("Electricity",720,1.287729,.8602),
        ("Traffic",96,1.973787,1.4696),("Traffic",192,2.004524,1.4508),("Traffic",336,2.063635,1.5147),("Traffic",720,2.166454,1.7351),
        ("Exchange",96,.213266,.1652),("Exchange",192,.423408,.4116),("Exchange",336,.764282,.8094),("Exchange",720,1.886809,1.7770),
        ("Solar",96,.588674,.5087),("Solar",192,.597890,.5406),("Solar",336,.623836,.5701),("Solar",720,.638530,.5507),
    ], columns=["dataset", "horizon", "pearson", "prr"])

baseline = ref.merge(l2, on=["dataset", "horizon"], how="inner")
baseline["horizon"] = baseline["horizon"].astype(int)
assert len(baseline) == 24


In [15]:
# Cell 15 — optional CRAFT-Graph reference from Exp47
craft_path = PROJECT_ROOT / "results" / "exp47_craft_matched_retrieval" / "craft_matched_exact_24.csv"

if craft_path.exists():
    craft = pd.read_csv(craft_path)
    keep = ["dataset","horizon"]
    if "craft_graph" in craft.columns:
        keep.append("craft_graph")
    craft = craft[keep]
    print("[FOUND] CRAFT matched results:", craft_path)
else:
    craft = None
    print("[INFO] CRAFT matched result CSV not found; RAFT analysis will proceed without it.")


[FOUND] CRAFT matched results: /data/code/which_histories_matter_ext/results/exp47_craft_matched_retrieval/craft_matched_exact_24.csv


In [16]:
# Cell 16 — exact matched comparison
raft_test = test_sel[[
    "dataset","horizon","raft_ms_afmse","raft_valscale_afmse","selected_g",
    "mean_multiscale_union"
]].copy()

comp = baseline.merge(raft_test, on=["dataset","horizon"], how="inner")

if craft is not None and "craft_graph" in craft.columns:
    comp = comp.merge(craft, on=["dataset","horizon"], how="left")

assert len(comp) == 24

def gain(method, baseline_name):
    return 100.0 * (comp[baseline_name] - comp[method]) / comp[baseline_name]

def wtl(x, tol=1e-12):
    x = np.asarray(x, float)
    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
        "mean_gain_pct": float(x.mean()),
        "median_gain_pct": float(np.median(x)),
    }

summary_rows = []

for method, label in [
    ("raft_ms_afmse", "RAFT-MS@10"),
    ("raft_valscale_afmse", "RAFT-ValScale@10"),
]:
    for b, blabel in [
        ("pearson", "Pearson"),
        ("anchored_l2", "Anchored L2"),
        ("prr", "PRR"),
    ]:
        summary_rows.append({
            "comparison": f"{label} vs {blabel}",
            **wtl(gain(method, b)),
        })

if "craft_graph" in comp.columns and comp["craft_graph"].notna().all():
    for method, label in [
        ("raft_ms_afmse", "RAFT-MS@10"),
        ("raft_valscale_afmse", "RAFT-ValScale@10"),
    ]:
        summary_rows.append({
            "comparison": f"{label} vs CRAFT-Graph@10",
            **wtl(gain(method, "craft_graph")),
        })

summary = pd.DataFrame(summary_rows)

display(comp.sort_values(["dataset","horizon"]))
display(summary)

comp.to_csv(OUT_ROOT / "raft_matched_exact_24.csv", index=False)
summary.to_csv(OUT_ROOT / "raft_matched_summary.csv", index=False)


,dataset,horizon,pearson,prr,anchored_l2,raft_ms_afmse,raft_valscale_afmse,selected_g,mean_multiscale_union,craft_graph
0,ETTh1,96,1.612267,1.1643,1.100985,1.439627,1.417994,1,32.119530,1.610380
1,ETTh1,192,1.817531,1.2229,1.277228,1.613701,1.602331,1,32.081253,1.774287
2,ETTh1,336,1.992872,1.4816,1.446432,1.733000,1.741645,1,32.012832,1.901292
3,ETTh1,720,2.254092,1.4618,1.714012,1.978460,1.980019,1,32.006944,2.174629
4,Electricity,96,1.179589,0.7191,0.703011,1.195281,1.198295,2,29.244686,1.186141
5,Electricity,192,1.195698,0.7345,0.728439,1.232099,1.231629,2,29.278373,1.180197
6,Electricity,336,1.241844,0.7648,0.768702,1.275571,1.277479,2,29.282733,1.229954
7,Electricity,720,1.287729,0.8602,0.815970,1.326263,1.327164,2,29.200628,1.270072
8,Exchange,96,0.213266,0.1652,0.186254,0.206670,0.206644,1,24.200422,0.228074
9,Exchange,192,0.423408,0.4116,0.379211,0.384629,0.384415,1,24.296380,0.501801


,comparison,W,T,L,mean_gain_pct,median_gain_pct
0,RAFT-MS@10 vs Pearson,13,0,11,7.570427,1.123346
1,RAFT-MS@10 vs Anchored L2,4,0,20,-27.992632,-26.497040
2,RAFT-MS@10 vs PRR,2,0,22,-29.545637,-26.604275
3,RAFT-ValScale@10 vs Pearson,15,0,9,7.904704,1.607599
4,RAFT-ValScale@10 vs Anchored L2,4,0,20,-27.543436,-25.424514
5,RAFT-ValScale@10 vs PRR,2,0,22,-29.220185,-26.083322
6,RAFT-MS@10 vs CRAFT-Graph@10,14,0,10,3.925778,9.035765
7,RAFT-ValScale@10 vs CRAFT-Graph@10,14,0,10,4.183701,8.672944


In [17]:
# Cell 17 — dataset-level comparison
dataset_rows = []

for d, g in comp.groupby("dataset", sort=False):
    row = {
        "dataset": d,
        "conditions": len(g),
        "selected_g_counts": json.dumps(
            g["selected_g"].value_counts().sort_index().to_dict()
        ),
        "mean_multiscale_union": float(g["mean_multiscale_union"].mean()),
    }

    for method, short in [
        ("raft_ms_afmse", "MS"),
        ("raft_valscale_afmse", "ValScale"),
    ]:
        for b, bshort in [
            ("pearson", "Pearson"),
            ("anchored_l2", "L2"),
            ("prr", "PRR"),
        ]:
            x = 100.0 * (g[b] - g[method]) / g[b]
            s = wtl(x)
            row[f"{short}_vs_{bshort}_W"] = s["W"]
            row[f"{short}_vs_{bshort}_T"] = s["T"]
            row[f"{short}_vs_{bshort}_L"] = s["L"]
            row[f"{short}_vs_{bshort}_mean_pct"] = s["mean_gain_pct"]

    dataset_rows.append(row)

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)
dataset_summary.to_csv(OUT_ROOT / "raft_matched_dataset_summary.csv", index=False)


,dataset,conditions,selected_g_counts,mean_multiscale_union,MS_vs_Pearson_W,MS_vs_Pearson_T,MS_vs_Pearson_L,MS_vs_Pearson_mean_pct,MS_vs_L2_W,MS_vs_L2_T,...,ValScale_vs_Pearson_L,ValScale_vs_Pearson_mean_pct,ValScale_vs_L2_W,ValScale_vs_L2_T,ValScale_vs_L2_L,ValScale_vs_L2_mean_pct,ValScale_vs_PRR_W,ValScale_vs_PRR_T,ValScale_vs_PRR_L,ValScale_vs_PRR_mean_pct
0,ETTh1,4,"{""1"": 4}",32.055140,4,0,0,11.797667,0,0,...,0,12.163767,0,0,4,-22.544064,0,0,4,-26.454737
1,Electricity,4,"{""2"": 4}",29.251605,0,0,4,-2.520731,0,0,...,4,-2.630699,0,0,4,-67.091243,0,0,4,-63.910202
2,Exchange,4,"{""1"": 4}",24.147313,4,0,0,6.648412,2,0,...,0,6.596583,2,0,2,-2.410435,2,0,2,-1.389487
3,Solar,4,"{""4"": 4}",22.693132,0,0,4,-2.504260,0,0,...,3,-1.670809,0,0,4,-20.471192,0,0,4,-14.827109
4,Traffic,4,"{""1"": 4}",27.936719,1,0,3,-1.414847,0,0,...,2,0.066588,0,0,4,-41.758149,0,0,4,-33.309871
5,Weather,4,"{""1"": 3, ""2"": 1}",30.413440,4,0,0,33.416320,2,0,...,0,32.902794,2,0,2,-10.985535,0,0,4,-35.429703


In [18]:
# Cell 18 — paper-facing block
print("\n" + "=" * 112)
print("PAPER-FACING RAFT MATCHED RETRIEVAL RESULT")
print("=" * 112)
print(
    f"Protocol: L={L}, stride={MEMORY_STRIDE}, TopK={TOP_K}; "
    f"official RAFT periods={RAFT_PERIODS}, topm={RAFT_TOPM}, temp={RAFT_TEMPERATURE}"
)
print("All six current-paper datasets are included in RAFT's original 10-dataset evaluation.")
print("IMPORTANT: matched retrieval-rule diagnostic, not end-to-end RAFT forecasting reproduction.")

print("\nOverall 24-condition comparisons")
print("--------------------------------")
for r in summary.itertuples(index=False):
    print(
        f"{r.comparison:38s}: "
        f"{r.W}W/{r.T}T/{r.L}L | "
        f"mean={r.mean_gain_pct:+.3f}% | median={r.median_gain_pct:+.3f}%"
    )

print("\nValidation-selected period counts:")
print(
    test_sel["selected_g"]
      .value_counts()
      .sort_index()
      .rename_axis("g")
      .to_string()
)

print(f"\nMean RAFT multiscale Top-20 union size: {comp['mean_multiscale_union'].mean():.1f}")

print("\nSaved under:", OUT_ROOT)
print("Files:")
for p in sorted(OUT_ROOT.glob("*.csv")):
    print(" -", p.name)



PAPER-FACING RAFT MATCHED RETRIEVAL RESULT
Protocol: L=96, stride=8, TopK=10; official RAFT periods=(4, 2, 1), topm=20, temp=0.1
All six current-paper datasets are included in RAFT's original 10-dataset evaluation.
IMPORTANT: matched retrieval-rule diagnostic, not end-to-end RAFT forecasting reproduction.

Overall 24-condition comparisons
--------------------------------
RAFT-MS@10 vs Pearson                 : 13W/0T/11L | mean=+7.570% | median=+1.123%
RAFT-MS@10 vs Anchored L2             : 4W/0T/20L | mean=-27.993% | median=-26.497%
RAFT-MS@10 vs PRR                     : 2W/0T/22L | mean=-29.546% | median=-26.604%
RAFT-ValScale@10 vs Pearson           : 15W/0T/9L | mean=+7.905% | median=+1.608%
RAFT-ValScale@10 vs Anchored L2       : 4W/0T/20L | mean=-27.543% | median=-25.425%
RAFT-ValScale@10 vs PRR               : 2W/0T/22L | mean=-29.220% | median=-26.083%
RAFT-MS@10 vs CRAFT-Graph@10          : 14W/0T/10L | mean=+3.926% | median=+9.036%
RAFT-ValScale@10 vs CRAFT-Graph@10    : 1

## Manuscript interpretation guide

The preferred reporting order is:

- **RAFT-MS@10** as the primary matched projection of RAFT's official multiscale retrieval;
- **RAFT-ValScale@10** as a stronger validation-selected control showing that the conclusion is
  not an artifact of how the three RAFT scales are collapsed into one Top-10 list.

Do not write that this reproduces RAFT's forecasting numbers.  The official RAFT system uses the
retrieved multiscale predictions inside its forecasting architecture.  This notebook isolates the
retrieval rule under the current paper's frozen history-relevance metric.

If PRR is stronger than both RAFT-MS and RAFT-ValScale on most conditions, the reviewer-facing
claim can be:

> Pearson is a controlled anchor rather than the sole baseline.  Under the same frozen retrieval
> protocol, we additionally compare with the retrieval rules of RAFT, SARAF, and CRAFT, while
> methods whose principal contribution is downstream semantic generation, cross-attention, or
> retrieval consumption are treated as complementary forecasting systems rather than as directly
> comparable standalone retrievers.

If RAFT is competitive or stronger on some domains, report those domains explicitly; this remains
consistent with the paper's conclusion that no single past-only retrieval rule dominates everywhere.
